In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from src.evaluation import load_train, cross_validate, poisson_deviance, PROCESSED, N_FOLDS
from src import gbm

train = load_train()
masks = [(train["Fold"] == f).to_numpy() for f in range(N_FOLDS)]
CHECKPOINTS = [100, 200, 300, 500, 700, 1000]
scores = np.zeros((len(CHECKPOINTS), N_FOLDS))

for f in range(N_FOLDS):
    tr, va = train[~masks[f]], train[masks[f]]
    model, levels = gbm.fit(tr, {**gbm.PARAMS, "n_estimators": max(CHECKPOINTS)}, gbm.FEATURES_V2)
    X_va = gbm.make_X(va, levels, gbm.FEATURES_V2)
    for j, k in enumerate(CHECKPOINTS):
        mu = model.predict(X_va, num_iteration=k) * va["Exposure"].to_numpy()
        scores[j, f] = poisson_deviance(va["ClaimNb"], mu)

table = pd.DataFrame(scores, index=CHECKPOINTS, columns=[f"fold {f}" for f in range(N_FOLDS)])
table["mean"] = scores.mean(axis=1)
print(table.round(4).to_string())

oof_v2, s_v2 = cross_validate(train, gbm.fit_predict_v2)      # official v2 score, 700 trees
print("\nGBM v2:", s_v2.round(4), "| mean", round(s_v2.mean(), 4))
pd.DataFrame({"IDpol": train["IDpol"], "pred_claims": oof_v2}).to_csv(PROCESSED / "oof_gbm_v2.csv", index=False)

      fold 0  fold 1  fold 2  fold 3  fold 4    mean
100   0.2923  0.2939  0.2932  0.2983  0.2906  0.2937
200   0.2904  0.2921  0.2915  0.2966  0.2887  0.2918
300   0.2898  0.2914  0.2909  0.2961  0.2880  0.2912
500   0.2892  0.2909  0.2904  0.2959  0.2876  0.2908
700   0.2890  0.2908  0.2902  0.2958  0.2874  0.2907
1000  0.2889  0.2908  0.2901  0.2958  0.2873  0.2906

GBM v2: [0.289  0.2908 0.2902 0.2958 0.2874] | mean 0.2907
